# GeoGuessr 單張街景 QLoRA

用同一個 Qwen3-VL-2B-Instruct 和同一個 prompt，比較 zero-shot 與一輪監督微調。訓練目標是國家碼和經緯度。Cross-entropy 只計算回答 token，GPS 距離用於評分。

在 VS Code 開啟 SSD 專案，再將 `geoguessr_sft_upload.zip` 上傳到 Colab 的 `/content`。選 GPU runtime 後執行以下各格。執行結束時，在 Colab Explorer 把結果 ZIP 存回專案的 `results/`，核對後終止 runtime。

## 2026-11-20 擴充到 6,000 張的 T4 實測

沿用同一個基底和 QLoRA 設定，訓練圖包含原本 2,000 張及新增的 4,000 張，共一輪 750 次更新。原來的 300 張評估圖與兩組既有預測均逐筆核對。正式訓練耗時 61 分 15 秒。

| 指標 | Zero-shot | 2,000 張 SFT | 6,000 張 SFT |
|---|---:|---:|---:|
| 國家猜對 | 132/300（44.0%） | 154/300（51.3%） | 156/300（52.0%） |
| 25 km 內 | 4/300（1.3%） | 9/300（3.0%） | 8/300（2.7%） |
| 200 km 內 | 32/300（10.7%） | 44/300（14.7%） | 44/300（14.7%） |
| 中位誤差（有效座標） | 1,141 km | 861 km | 832 km |

和 2,000 張版相比，25 km 內新增命中 1 張，原本命中的題目有 2 張在新版失誤。完整結果和最終 adapter 在 `results/qlora_6000/geoguessr-sft-6000-results.zip`。


## 2026-11-20 Colab T4 實測

Qwen3‑VL‑2B‑Instruct，4-bit QLoRA（rank 8），2,000 張訓練圖，一輪共 250 次更新。在同一批 300 張保留照片上，分別停用與啟用 adapter，使用相同的 prompt 和解碼設定。

| 指標 | Zero-shot | 一輪 SFT |
|---|---:|---:|
| 國家猜對 | 132/300（44.00%） | 154/300（51.33%） |
| 25 km 命中 | 4/300（1.33%） | 9/300（3.00%） |
| 200 km 命中 | 32/300（10.67%） | 44/300（14.67%） |
| 中位誤差（有效座標） | 1,141 km | 861 km |
| 有效座標 | 294/300（98.00%） | 300/300（100.00%） |

25 km 內新增命中 5 張，也有 0 張從命中變為失誤。完整結果和 adapter 已保存在 SSD 的 `results/qlora/geoguessr-sft-run-results.zip`。本次完整結果來自 Colab CLI 執行同一個訓練腳本，並已核對 GPU runtime 釋放。


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import shutil
import subprocess
import sys

ROOT = Path('/content/geoguessr_sft')
ROOT.mkdir(parents=True, exist_ok=True)
UPLOAD = Path('/content/geoguessr_sft_upload.zip')
PARTS = Path('/content/sft_upload_parts')
if PARTS.is_dir():
    temporary = UPLOAD.with_suffix('.part')
    with temporary.open('wb') as output:
        for part in sorted(PARTS.glob('part-*')):
            with part.open('rb') as source:
                shutil.copyfileobj(source, output)
    with temporary.open('rb') as handle:
        received = hashlib.file_digest(handle, 'sha256').hexdigest()
    if received != (PARTS / 'bundle.sha256').read_text().strip():
        raise ValueError('Upload checksum mismatch; confirm all parts have arrived.')
    temporary.replace(UPLOAD)
if not UPLOAD.is_file():
    raise FileNotFoundError(f'Upload {UPLOAD.name} or {PARTS.name} first.')
with ZipFile(UPLOAD) as archive:
    for name in ('phase1.py', 'sft_colab.py', 'requirements-colab.txt'):
        (ROOT / name).write_bytes(archive.read(name))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                '-r', str(ROOT / 'requirements-colab.txt')], check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU kernel before training.")
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

In [ ]:
# Smoke check first, then run the full subset using the already-cached model.
for mode in ("smoke", "run"):
    command = [sys.executable, "-u", str(ROOT / "sft_colab.py"), str(UPLOAD),
               "--output", str(ROOT / mode),
               "--export", f"/content/geoguessr-sft-{mode}-results.zip"]
    if mode == "smoke":
        command.append("--smoke")
    with (ROOT / f"{mode}.log").open("a") as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line.rstrip(), flush=True)
            log.write(line)
            log.flush()
        if (status := proc.wait()) != 0:
            raise subprocess.CalledProcessError(status, command)

In [ ]:
import pandas as pd

metrics = pd.read_csv(ROOT / "run" / "metrics.csv")
for column in ("valid_coordinates", "country_correct", "within_25km", "within_200km"):
    metrics[f"{column}_pct"] = (100 * metrics[column] / metrics["n"]).round(2)
display(metrics)
print((ROOT / "run" / "paired_metrics.json").read_text())
print("Download /content/geoguessr-sft-run-results.zip to the SSD before terminating the runtime.")